# Streaming Fraud Detection — Kafka + Online Feature Store

This notebook demonstrates the serving architecture without requiring a GPU or a permanent Kafka cluster. Kafka code is provided for a real deployment, while the local simulator makes the notebook runnable on a normal laptop.


In [1]:
from pathlib import Path
import json
import sqlite3
import time
import uuid
from datetime import datetime
import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_PATH = ROOT / 'models' / 'fraud_model.joblib'
FEATURE_DB = ROOT / 'models' / 'feature_store.db'
bundle = joblib.load(MODEL_PATH)
model = bundle['model']
features = bundle['features']


## 1. Online feature store

A feature store separates the storage of reusable model features from the model itself. SQLite is used here only because it is zero-configuration and laptop-friendly. In production this could be Redis, Feast or a managed online store.


In [2]:
def get_features(user_id):
    conn = sqlite3.connect(FEATURE_DB)
    row = conn.execute(
        'SELECT user_tx_count_24h, amount_avg_24h, device_seen_count FROM online_features WHERE user_id = ?',
        (user_id,)
    ).fetchone()
    conn.close()
    if row is None:
        return {'user_tx_count_24h': 0, 'amount_avg_24h': 0.0, 'device_seen_count': 0}
    return {'user_tx_count_24h': row[0], 'amount_avg_24h': row[1], 'device_seen_count': row[2]}

get_features('demo_user_001')


{'user_tx_count_24h': 2, 'amount_avg_24h': 48.5, 'device_seen_count': 2}

## 2. Build an event and score it

A streaming event contains only the information available when the transaction arrives. Historical behaviour comes from the online feature store.


In [5]:
event = {
    'transaction_id': str(uuid.uuid4()),
    'user_id': 'demo_user_001',
    'purchase_value': 125.0,
    'age': 34,
    'hour': 23,
    'dayofweek': 5,
    'new_device': 1,
    'high_risk_source': 1,
    'device_seen_count': 0,
}

online = get_features(event['user_id'])
row = {
    'purchase_value': event['purchase_value'],
    'age': event['age'],
    'hour': event['hour'],
    'dayofweek': event['dayofweek'],
    'user_tx_count_24h': online['user_tx_count_24h'],
    'device_seen_count': event['device_seen_count'],
    'amount_avg_24h': online['amount_avg_24h'],
    'velocity_score': min(1.0, online['user_tx_count_24h'] / 10),
    'new_device': event['new_device'],
    'high_risk_source': event['high_risk_source'],
}
feature_row = {col: 0.0 for col in features}
feature_row.update({
    'Amount': float(event['purchase_value']),
    'amount_sq': float(event['purchase_value']) ** 2,
    'abs_amount': abs(float(event['purchase_value'])),
    'amount_rank': 0.0,
    'hour': int(event['hour']),
    'dayofweek': int(event['dayofweek']),
    'account_age_days': 0.0,
})
probability = model.predict_proba(pd.DataFrame([feature_row], columns=features))[:, 1][0]
print(json.dumps({'transaction_id': event['transaction_id'], 'fraud_probability': round(float(probability), 4)}, indent=2))


{
  "transaction_id": "e2ab6759-4aee-44a8-9ca4-3c874c4b06b0",
  "fraud_probability": 0.0004
}


## 3. Kafka producer/consumer pattern

The following is the integration pattern. It is intentionally not executed unless a Kafka broker is running locally.


In [8]:
# Producer example:
# from kafka import KafkaProducer
# producer = KafkaProducer(
#     bootstrap_servers='localhost:9092',
#     value_serializer=lambda value: json.dumps(value).encode('utf-8')
# )
# producer.send('fraud-transactions', event)
# producer.flush()

# Consumer example:
# from kafka import KafkaConsumer
# consumer = KafkaConsumer(
#     'fraud-transactions',
#     bootstrap_servers='localhost:9092',
#     value_deserializer=lambda value: json.loads(value.decode('utf-8')),
#     auto_offset_reset='latest'
# )
# for message in consumer:
#     transaction = message.value
#     # Read online features -> build model row -> score -> publish decision
#     break


## 4. Laptop-friendly stream simulator

This keeps the real-time design understandable even when Kafka is not installed. The same event object can later be replaced by a Kafka message.


In [10]:
events = [
    {'user_id': 'demo_user_001', 'purchase_value': 35, 'new_device': 0, 'high_risk_source': 0},
    {'user_id': 'demo_user_002', 'purchase_value': 280, 'new_device': 1, 'high_risk_source': 1},
    {'user_id': 'new_user_101', 'purchase_value': 60, 'new_device': 1, 'high_risk_source': 0},
]

for event in events:
    online = get_features(event['user_id'])
    row = {
        'purchase_value': event['purchase_value'], 'age': 35, 'hour': 23,
        'dayofweek': datetime.now().weekday(),
        'user_tx_count_24h': online['user_tx_count_24h'],
        'device_seen_count': online['device_seen_count'],
        'amount_avg_24h': online['amount_avg_24h'],
        'velocity_score': min(1.0, online['user_tx_count_24h'] / 10),
        'new_device': event['new_device'], 'high_risk_source': event['high_risk_source']
    }
    feature_row = {col: 0.0 for col in features}
    feature_row.update({
        'Amount': float(row['purchase_value']),
        'amount_sq': float(row['purchase_value']) ** 2,
        'abs_amount': abs(float(row['purchase_value'])),
        'amount_rank': 0.0,
        'hour': int(row['hour']),
        'dayofweek': int(row['dayofweek']),
        'account_age_days': 0.0,
    })
    score = float(model.predict_proba(pd.DataFrame([feature_row], columns=features))[0, 1])
    decision = 'FRAUD REVIEW' if score >= 0.50 else 'APPROVE'
    print(event['user_id'], '->', decision, f'({score:.2%})')
    time.sleep(0.5)


demo_user_001 -> APPROVE (0.04%)
demo_user_002 -> APPROVE (0.06%)
new_user_101 -> APPROVE (0.04%)


## Production extensions

At larger scale, the next engineering steps would be partitioning Kafka by user/card identifier, using a distributed online feature store, maintaining event-time windows, handling late events, registering model versions, monitoring feature drift, and deploying multiple scoring workers behind a load balancer.
